We use OBR's own forecast of housing starts/completions since it embeds the policy scenario

In [ ]:
import sys
sys.path.append("../..")

import pandas as pd
import numpy as np
import json
import matplotlib.pyplot as plt
import scienceplots
import openpyxl
from matplotlib.ticker import MaxNLocator
from python.functions.bridge import build_bridge_inputs, run_bridge

In [ ]:
RAW_DIR = "../../data/raw"
OUT_DIR = "../../data/outputs"

B = build_bridge_inputs()

wb = openpyxl.load_workbook(
    f"{RAW_DIR}/OBR/efo-march-2026-detailed-forecast-tables-economy.xlsx",
    read_only=True, data_only=True
)
ws = wb["1.16"]
rows = list(ws.iter_rows(values_only=True))

recs = [(r[1], r[5]) for r in rows if r[1] and len(str(r[1])) == 6 and str(r[1])[4] == "Q"]
obr = pd.DataFrame(recs, columns=["period", "starts_uk"])
obr["period"] = pd.PeriodIndex(obr["period"], freq="Q")
obr = obr[obr["period"] >= "2026Q1"]  # check this matches your ensemble/VECM CSVs' start quarter

ENGLAND_SHARE = 0.846
obr["starts_eng"] = obr["starts_uk"] * ENGLAND_SHARE
obr["ensemble_log"] = np.log(obr["starts_eng"])
obr_out = obr[["period", "ensemble_log"]].copy()
obr_out["period"] = obr_out["period"].astype(str)

obr_out.to_csv(f"{OUT_DIR}/OBR/obr_own_starts_scenario.csv", index=False)
obr_out.head()

In [ ]:
ardl  = run_bridge(f"{OUT_DIR}/forecasts/obr_scenario_forecasts.csv", "ardl_log",
                         B["model"], B["smearing"], B["hist_ln_C"], B["hist_ln_S"],
                         B["fy_map"], B["net_add"], B["actual_back"])
vecm       = run_bridge(f"{OUT_DIR}/forecasts/vecm_unconditional_forecast.csv", "vecm_log",
                         B["model"], B["smearing"], B["hist_ln_C"], B["hist_ln_S"],
                         B["fy_map"], B["net_add"], B["actual_back"], strip_space=True)
obr_reform = run_bridge(f"{OUT_DIR}/OBR/obr_own_starts_scenario.csv", "ensemble_log",
                         B["model"], B["smearing"], B["hist_ln_C"], B["hist_ln_S"],
                         B["fy_map"], B["net_add"], B["actual_back"])

target = 1_500_000
for name, d in [("ARDL (baseline)", ardl),
                ("VECM (unconditional)", vecm),
                ("OBR own starts forecast (reform-inclusive)", obr_reform)]:
    print(name)
    for fy, v in d.items():
        print(f"{fy}: {v:,.0f}")
    cumulative = sum(d.values())
    print(f"Cumulative: {cumulative:,.0f} ({100*cumulative/target:.1f}% of {target:,}, "
          f"shortfall {target-cumulative:,.0f})\n")

In [ ]:
fy_start = lambda s: int(str(s)[:4])

actual = B["lt120"]["Total net additional dwellings"].dropna()
actual.index = actual.index.map(fy_start)

# Backfill both known-but-missing years: 2024-25 and 2025-26
if 2024 not in actual.index:
    actual.loc[2024] = ardl["2024-25"]
if 2025 not in actual.index:
    actual.loc[2025] = ardl["2025-26"]  # identical across models — this is known, not forecast
actual = actual.sort_index()

LAST_ACTUAL_YEAR = 2025  # 2025-26 is the last known year
TARGET_START_YEAR = 2024  # FY2024-25 is year 1 of the 5-year target window
TARGET_END_YEAR = 2028    # FY2028-29 is year 5

def cum_series(d):
    # Forecast years strictly after the last known year
    fcast = pd.Series({fy_start(k): v for k, v in d.items() if fy_start(k) > LAST_ACTUAL_YEAR}).sort_index()
    # Anchor on the known actuals (2024-25, 2025-26), then stack forecast years on top
    combined = pd.concat([actual.loc[[TARGET_START_YEAR, LAST_ACTUAL_YEAR]], fcast])
    return combined.cumsum()

cum_rdl  = cum_series(ardl)
cum_vecm = cum_series(vecm)
cum_obr  = cum_series(obr_reform)

# Straight-line target: 300,000/year average pace, 1.5m cumulative by year 5
years = list(range(TARGET_START_YEAR, TARGET_END_YEAR + 1))
year_number = {y: i + 1 for i, y in enumerate(years)}  # 2024→1, ..., 2028→5
target_cum = pd.Series({y: 300_000 * n for y, n in year_number.items()})

with plt.style.context(["science", "no-latex", "high-vis"]):
    fig, ax = plt.subplots(figsize=(7, 4))

    # Target gets the same "non-cycle" treatment as before: distinct, deliberately plain
    ax.plot(target_cum.index, target_cum.values, ls="--", color="0.5", lw=1.4,
             label="1.5m target pace", zorder=1)

    # Forecasts pull color+ls+marker from the high-vis cycle, fixed meaningful order
    for s, label, lw, z in [(cum_rdl, "ARDL", 1.6, 3), (cum_vecm, "VECM", 1.6, 3), (cum_obr, "OBR own forecast (reform-inclusive)", 2.2, 4)]:
        ax.plot(s.index, s.values, lw=lw, label=label, zorder=z)

    ax.fill_between(cum_rdl.index, cum_rdl.values, target_cum.reindex(cum_rdl.index).values,
                 color="0.85", alpha=0.5, zorder=0)
    ax.set_xlim(TARGET_START_YEAR, TARGET_END_YEAR)
    ax.xaxis.set_major_locator(MaxNLocator(integer=True))
    ax.set_xticks(years)
    ax.set_xticklabels([f"{y}/{str(y+1)[-2:]}" for y in years])
    ax.set_ylabel("Cumulative net additional dwellings")
    ax.set_xlabel("Financial year (start)")
    ax.legend(loc="lower right", frameon=False)
    fig.tight_layout()
    fig.savefig("../../data/outputs/figures/obr_projection_cumulative.png", dpi=300)
    plt.show()